# Climate Resilience and Agricultural Technology Adoption Among Smallholder Farmers in Kenya

## Objective

This project investigates the factors influencing adoption of modern agricultural practices among smallholder farmers and evaluates how climate risks, financial inclusion, and farmer characteristics affect agricultural resilience.

This analysis is structured to reflect rigorous experimental and data science standards relevant to agricultural development programmes:

1. **Causal inference framing** – distinguishing association from causation, with an RCT design proposal
2. **Mixed-effects modelling** – accounting for regional clustering in farmer data
3. **Observational confounding** – identifying and controlling for selection bias in financing access
4. **Climate vulnerability segmentation** – identifying high-risk farmer profiles for targeted intervention
5. **Reproducible ML pipeline** – feature encoding, model evaluation, and actionable recommendations

## Research Questions

1. Does access to financing increase adoption of modern farming practices?
2. Which farmer groups are most vulnerable to climate-related losses?
3. What factors predict adoption of improved agricultural practices?
4. How can data-driven recommendations improve agricultural resilience?

> **Methodological note:** This is a cross-sectional observational dataset. Associations identified here are *correlational*, not causal. Section 5 includes a formal discussion of confounders and proposes an RCT design to establish causal estimates.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Visualisation
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

# Statistical inference
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal
import statsmodels.formula.api as smf          # mixed-effects models
import statsmodels.api as sm

# Machine learning
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (classification_report, roc_auc_score,
                              ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All libraries loaded successfully.")

## 2. Load Data

In [ ]:
df = pd.read_csv("Agri.csv")

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
# Missing value audit
missing = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Values'] > 0].sort_values('Percentage (%)', ascending=False)
print(missing if not missing.empty else "No missing values detected.")

### Initial Observations

| Category | Variables present |
|---|---|
| Farmer demographics | region, education level, gender, age group |
| Land & experience | land tenure, farming experience, farm size |
| Financial inclusion | agricultural financing, bank account, mobile account |
| Technology access | phone ownership, internet use |
| Agronomic practices | fertiliser, certified seeds, pest management |
| Climate loss exposure | losses-rain pattern, drought, heatwave, storms, mudslides |

All categorical variables will be imputed with the column mode; continuous variables with the median.

## 3. Data Cleaning

In [ ]:
# ── Categorical imputation ─────────────────────────────────────────────────
categorical_cols = [
    'region', 'education level', 'gender', 'age group', 'land tenure',
    'phone ownership', 'internet use', 'agricultural financing',
    'bank account', 'mobile account', 'farmer organization',
    'service provider awareness', 'agricultural continuity', 'land expansion'
]

for col in categorical_cols:
    if col in df.columns:
        mode_val = df[col].mode()[0]
        n_filled = df[col].isnull().sum()
        df[col].fillna(mode_val, inplace=True)
        if n_filled:
            print(f"  '{col}': filled {n_filled} missing → '{mode_val}'")

print("Categorical imputation complete.")

In [ ]:
# ── Farming experience: extract numeric years ─────────────────────────────
df['farming experience'] = (
    df['farming experience']
    .astype(str)
    .str.extract(r'(\d+)')
    .astype(float)
)
df['farming experience'].fillna(df['farming experience'].median(), inplace=True)
print(f"Farming experience – median: {df['farming experience'].median():.1f} yrs, "
      f"range: {df['farming experience'].min():.0f}–{df['farming experience'].max():.0f} yrs")

In [ ]:
# ── Farm size: extract numeric acres, then drop (high missingness) ─────────
df['farm size (acres)'] = (
    df['farm size (acres)']
    .astype(str)
    .str.extract(r'(\d+\.?\d*)')
    .astype(float)
)
df['farm size (acres)'].fillna(df['farm size (acres)'].median(), inplace=True)
print(df['farm size (acres)'].describe())

# Drop: low signal-to-noise after imputation
df = df.drop(columns=['farm size (acres)'])
print("\n'farm size (acres)' dropped after review.")

In [ ]:
# ── Agronomic practice variables ──────────────────────────────────────────
practice_cols = ['fertiliser', 'certified seeds', 'pest management']
for col in practice_cols:
    if col in df.columns:
        df[col].fillna(df[col].mode()[0], inplace=True)

# ── Climate loss variables: standardise to lowercase ──────────────────────
risk_raw = ['losses-rain pattern', 'losses-drought', 'losses-heatwave',
            'losses-storms', 'losses-mudslides']
for col in risk_raw:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.lower()

print("Remaining missing values:")
remaining = df.isnull().sum()
print(remaining[remaining > 0] if remaining.any() else "  None – dataset is clean.")

## 4. Feature Engineering

### 4.1 Technology Adoption Score

We define a composite **adoption score** from three binary practice indicators:
- Fertiliser use
- Certified seed use  
- Pest management

A farmer with score ≥ 2 is classified as **high adopter** — i.e. they have adopted a majority of the recommended practices.

> **Design note:** A threshold of 2/3 was chosen rather than 3/3 to avoid penalising farmers who may rationally skip one practice due to crop type or agro-ecological context. Sensitivity to this threshold is discussed in Section 7.

In [ ]:
df["fertiliser_score"] = df["fertiliser"].map({"yes": 1, "no": 0})
df["seed_score"]       = df["certified seeds"].map({"yes": 1, "no": 0})
df["pest_score"]       = df["pest management"].map({"yes": 1, "no": 0})

df["adoption_score"] = df["fertiliser_score"] + df["seed_score"] + df["pest_score"]

df["high_adoption"] = (df["adoption_score"] >= 2).astype(int)

print("Adoption score distribution:")
print(df["adoption_score"].value_counts().sort_index())
print(f"\nHigh adopters (score ≥ 2): {df['high_adoption'].mean()*100:.1f}% of farmers")

### 4.2 Climate Risk Score

Binary loss indicators for five hazard types are summed to a **climate risk score** (0–5).
Higher scores indicate farmers exposed to a wider portfolio of climate shocks.

In [ ]:
risk_cols = ['losses-rain pattern', 'losses-drought', 'losses-heatwave',
            'losses-storms', 'losses-mudslides']

for col in risk_cols:
    df[col] = df[col].map({"yes": 1, "no": 0, "1": 1, "0": 0}).fillna(0).astype(int)

df["climate_risk_score"] = df[risk_cols].sum(axis=1)

print("Climate risk score distribution:")
print(df["climate_risk_score"].value_counts().sort_index())
print(f"\nMean climate risk score: {df['climate_risk_score'].mean():.2f} / 5")

## 5. Exploratory Data Analysis

### 5.1 Adoption Rates by Demographic Group

We examine how technology adoption varies across education, gender, and region.
These patterns inform targeting strategies for extension services.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Education
edu_order = ['no formal education', 'primary', 'secondary', 'tertiary']
edu_order = [e for e in edu_order if e in df['education level'].unique()]
adoption_by_edu = (df.groupby('education level')['high_adoption']
                   .mean().mul(100).reindex(edu_order))
axes[0].bar(adoption_by_edu.index, adoption_by_edu.values, color=sns.color_palette("Blues_d", len(adoption_by_edu)))
axes[0].set_title('Adoption Rate by Education Level', fontweight='bold')
axes[0].set_ylabel('High Adoption Rate (%)')
axes[0].set_ylim(0, 100)
axes[0].tick_params(axis='x', rotation=30)
for i, v in enumerate(adoption_by_edu.values):
    axes[0].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

# Gender
adoption_by_gender = df.groupby('gender')['high_adoption'].mean().mul(100)
axes[1].bar(adoption_by_gender.index, adoption_by_gender.values, color=['#5B8DB8', '#E8927C'])
axes[1].set_title('Adoption Rate by Gender', fontweight='bold')
axes[1].set_ylabel('High Adoption Rate (%)')
axes[1].set_ylim(0, 100)
for i, v in enumerate(adoption_by_gender.values):
    axes[1].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

# Financing
adoption_by_fin = df.groupby('agricultural financing')['high_adoption'].mean().mul(100)
axes[2].bar(adoption_by_fin.index, adoption_by_fin.values, color=['#D16A5B', '#5BA85B'])
axes[2].set_title('Adoption Rate by Financing Access', fontweight='bold')
axes[2].set_ylabel('High Adoption Rate (%)')
axes[2].set_ylim(0, 100)
for i, v in enumerate(adoption_by_fin.values):
    axes[2].text(i, v + 1, f'{v:.1f}%', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('adoption_by_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved.")

### 5.2 Climate Risk Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of climate risk scores
sns.histplot(df["climate_risk_score"], bins=6, ax=axes[0],
             kde=False, color='steelblue', edgecolor='white')
axes[0].set_title("Distribution of Climate Risk Score", fontweight='bold')
axes[0].set_xlabel("Number of Climate Hazards Experienced")
axes[0].set_ylabel("Number of Farmers")

# Climate risk score vs adoption
sns.boxplot(data=df, x="climate_risk_score", y="adoption_score",
            ax=axes[1], palette="Blues")
axes[1].set_title("Adoption Score by Climate Risk Exposure", fontweight='bold')
axes[1].set_xlabel("Climate Risk Score")
axes[1].set_ylabel("Technology Adoption Score")

plt.tight_layout()
plt.savefig('climate_risk_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

### 5.3 Regional Variation in Adoption and Climate Risk

In [ ]:
if 'region' in df.columns:
    regional = df.groupby('region').agg(
        adoption_rate=('high_adoption', 'mean'),
        avg_climate_risk=('climate_risk_score', 'mean'),
        n_farmers=('high_adoption', 'count')
    ).mul({'adoption_rate': 100, 'avg_climate_risk': 1, 'n_farmers': 1}).round(2)
    regional = regional.sort_values('adoption_rate', ascending=False)

    fig, ax = plt.subplots(figsize=(12, 5))
    x = np.arange(len(regional))
    bars = ax.bar(x, regional['adoption_rate'], color='steelblue', alpha=0.8, label='Adoption Rate (%)')
    ax2 = ax.twinx()
    ax2.plot(x, regional['avg_climate_risk'], 'o-', color='tomato', linewidth=2, label='Avg Climate Risk')
    ax.set_xticks(x)
    ax.set_xticklabels(regional.index, rotation=45, ha='right')
    ax.set_ylabel('High Adoption Rate (%)')
    ax2.set_ylabel('Average Climate Risk Score')
    ax.set_title('Adoption Rate and Climate Risk by Region', fontweight='bold')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right')
    plt.tight_layout()
    plt.savefig('regional_adoption_risk.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(regional.to_string())

## 6. Statistical Impact Analysis

### 6.1 Financing and Technology Adoption – Chi-Square Test

**Null hypothesis H₀:** Agricultural financing access is independent of technology adoption status.

**Test:** Pearson chi-square test on the 2×2 contingency table.

In [ ]:
table = pd.crosstab(df["agricultural financing"], df["high_adoption"],
                   rownames=["Financing"], colnames=["High Adoption"])
print("Contingency table:")
print(table)

chi2, p, dof, expected = chi2_contingency(table)

# Effect size: Cramér's V
n = table.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(table.shape) - 1)))

print(f"\nChi² = {chi2:.3f}, df = {dof}, p-value = {p:.4f}")
print(f"Cramér's V (effect size) = {cramers_v:.3f}")
print("\nInterpretation:")
if p < 0.05:
    strength = "small" if cramers_v < 0.1 else "moderate" if cramers_v < 0.3 else "large"
    print(f"  ✓ Significant association (p < 0.05). Effect size is {strength} (V={cramers_v:.3f}).")
    print(f"  Farmers with financing access are more likely to be high adopters.")
    print(f"  ⚠ This is an ASSOCIATION, not a causal estimate (see Section 6.3 on confounders).")
else:
    print("  ✗ No significant association detected at α = 0.05.")

### 6.2 Mixed-Effects Logistic Regression

A key limitation of simple logistic regression here is **regional clustering** — farmers in the same region share soil type, rainfall patterns, extension service access, and market proximity. Ignoring this inflates standard errors and can bias coefficient estimates.

We fit a **generalised linear mixed model (GLMM)** with:
- **Fixed effects:** financing, education, gender, internet use, farming experience, climate risk score
- **Random effect:** region (random intercept per region)

This accounts for the non-independence of observations within regions — a critical methodological requirement for programme data spanning multiple geographies.

In [ ]:
# Prepare modelling dataframe
model_df = df.copy()

# Encode binary predictors
binary_map = {'yes': 1, 'no': 0}
for col in ['agricultural financing', 'phone ownership', 'internet use',
            'bank account', 'mobile account']:
    if col in model_df.columns:
        model_df[col] = model_df[col].map(binary_map).fillna(0).astype(int)

# Encode education level ordinally
edu_map = {'no formal education': 0, 'primary': 1, 'secondary': 2, 'tertiary': 3}
if 'education level' in model_df.columns:
    model_df['education_ord'] = model_df['education level'].map(edu_map).fillna(0)

# Gender dummy
if 'gender' in model_df.columns:
    model_df['gender_male'] = (model_df['gender'].str.lower() == 'male').astype(int)

# Standardise continuous predictors (improves convergence)
for col in ['farming experience', 'climate_risk_score']:
    if col in model_df.columns:
        model_df[f'{col}_z'] = (
            (model_df[col] - model_df[col].mean()) / model_df[col].std()
        )

# ── Rename columns to be formula-safe (no spaces) ──────────────────────────
model_df = model_df.rename(columns={
    'agricultural financing': 'agri_financing',
    'internet use':           'internet_use',
    'phone ownership':        'phone_ownership',
})

# Ensure all formula columns are numeric float (statsmodels requires this)
formula_cols = ['high_adoption', 'agri_financing', 'education_ord',
                'gender_male', 'internet_use',
                'farming experience_z', 'climate_risk_score_z']

for col in formula_cols:
    if col in model_df.columns:
        model_df[col] = pd.to_numeric(model_df[col], errors='coerce').fillna(0).astype(float)

print("Model dataframe prepared.")
print(model_df[formula_cols].describe().round(3))

In [ ]:
# Mixed-effects linear probability model
# Random intercept by region accounts for unobserved regional heterogeneity
#
# Note: statsmodels MixedLM fits a LINEAR mixed model (LMM), not logistic.
# For a binary outcome this is a 'linear probability model' — coefficients
# are interpreted as percentage-point changes in probability of high adoption.
# This is standard practice when a full GLMM is not available in statsmodels.

fixed_terms = []
for col in ['agri_financing', 'education_ord', 'gender_male',
            'internet_use', 'farming experience_z', 'climate_risk_score_z']:
    if col in model_df.columns:
        fixed_terms.append(col)

formula_fixed = 'high_adoption ~ ' + ' + '.join(fixed_terms)
print(f"Formula: {formula_fixed}")

if 'region' in model_df.columns:
    # Ensure region has no NaN
    fit_df = model_df[fixed_terms + ['high_adoption', 'region']].dropna()

    try:
        md_model = smf.mixedlm(
            formula_fixed,
            data=fit_df,
            groups=fit_df['region']
        )
        md_result = md_model.fit(method='lbfgs', maxiter=500)
        print(md_result.summary())
        print("\n✓ Mixed-effects model converged successfully.")
        print("  Random effect (region) captures unobserved spatial heterogeneity.")

    except Exception as e:
        print(f"Mixed-effects model note: {e}")
        print("\nFalling back to OLS logistic regression with region dummies.")
        fit_df2 = pd.get_dummies(fit_df, columns=['region'], drop_first=True)
        # Ensure all dummy cols are float
        for c in fit_df2.columns:
            fit_df2[c] = pd.to_numeric(fit_df2[c], errors='coerce').fillna(0)
        X_ols = fit_df2.drop(columns=['high_adoption'])
        y_ols = fit_df2['high_adoption']
        lr = sm.Logit(y_ols, sm.add_constant(X_ols.astype(float))).fit(disp=False, maxiter=200)
        print(lr.summary2())
else:
    print("No 'region' column — fitting standard logistic regression.")
    fit_df = model_df[fixed_terms + ['high_adoption']].dropna()
    X_lr = fit_df[fixed_terms].astype(float)
    y_lr = fit_df['high_adoption'].astype(float)
    lr = sm.Logit(y_lr, sm.add_constant(X_lr)).fit(disp=False)
    print(lr.summary2())

### 6.3 Causal Inference: Confounders and RCT Design Proposal

> **Why this matters for programme decisions:** The chi-square test shows financing is *associated* with adoption, but we cannot conclude that providing financing *causes* higher adoption without accounting for selection bias.

#### Identified confounders

| Confounder | Direction of bias | Notes |
|---|---|---|
| Education level | Upward | More educated farmers more likely to both seek financing AND adopt technology |
| Region | Mixed | Some regions have better financing AND extension access |
| Land tenure | Upward | Formal tenure holders can access credit AND are more likely to invest in inputs |
| Farmer organisation membership | Upward | Group members access financing and get group extension services |
| Wealth / farm size | Upward | Wealthier farmers can self-finance inputs regardless |

#### Proposed RCT Design: Randomised Agricultural Financing Intervention

To obtain a clean causal estimate of financing on adoption:

**Design:** Cluster-randomised controlled trial (RCBD)
- **Unit of randomisation:** Village/ward (to avoid spillover contamination)
- **Stratification variables:** Region, baseline adoption rate, farmer organisation density
- **Treatment arm:** Access to subsidised seasonal input credit
- **Control arm:** Status quo (no financing access)
- **Primary outcome:** High adoption binary (fertiliser + certified seed + pest management)
- **Secondary outcomes:** Yield (kg/ha), income, climate resilience score
- **Follow-up period:** Two growing seasons (to capture learning effects)

**Power calculation framework:**
- Assumed baseline adoption rate (control): ~40%
- Minimum detectable effect (MDE): 10 percentage point increase
- Intracluster correlation (ICC): 0.05 (typical for agricultural trials)
- Power: 80%, α = 0.05
- Required sample: calculated below

In [ ]:
# Power calculation for cluster RCT
# Using Donner & Klar formula for cluster randomised trials

def cluster_rct_sample_size(p_control, mde, icc, cluster_size, power=0.80, alpha=0.05):
    """
    Calculate required number of clusters per arm for a cluster RCT.
    
    Parameters
    ----------
    p_control    : float – baseline proportion in control arm
    mde          : float – minimum detectable effect (absolute, percentage points)
    icc          : float – intracluster correlation coefficient
    cluster_size : int   – average number of farmers per cluster (village)
    power        : float – desired statistical power (default 0.80)
    alpha        : float – significance level (default 0.05)
    
    Returns
    -------
    n_clusters : int – clusters required per arm
    n_farmers  : int – total farmers across both arms
    """
    from scipy.stats import norm
    p_treat = p_control + mde
    p_bar   = (p_control + p_treat) / 2
    
    z_alpha = norm.ppf(1 - alpha / 2)
    z_beta  = norm.ppf(power)
    
    # Variance inflation factor (VIF) for clustering
    vif = 1 + (cluster_size - 1) * icc
    
    # Simple two-proportion z-test sample size, then inflate by VIF
    n_simple = (z_alpha + z_beta)**2 * (p_control*(1-p_control) + p_treat*(1-p_treat)) / mde**2
    n_per_cluster = n_simple * vif / cluster_size
    n_clusters = int(np.ceil(n_per_cluster))
    
    return n_clusters, n_clusters * 2 * cluster_size

for cluster_sz in [20, 30, 50]:
    clusters, total = cluster_rct_sample_size(
        p_control=0.40, mde=0.10, icc=0.05, cluster_size=cluster_sz
    )
    print(f"Cluster size = {cluster_sz} farmers: "
          f"{clusters} clusters/arm → {total:,} total farmers")

print("\n→ For a typical village of ~30 farmers, approximately 2×[clusters] villages required.")
print("  This is a practical scale for a One Acre Fund pilot programme.")

## 7. Climate Vulnerability Analysis

### 7.1 High-Risk Farmer Profile

We identify farmers with climate_risk_score ≥ 3 (exposed to 3 or more hazards) as **high-vulnerability** and compare their adoption rates, financing access, and demographics against low-risk farmers.

In [ ]:
df['high_climate_risk'] = (df['climate_risk_score'] >= 3).astype(int)

print("High vs Low Climate Risk – Adoption and Financing Rates:")
print("=" * 60)

for grp_name, grp_df in df.groupby('high_climate_risk'):
    label = "HIGH risk (≥3 hazards)" if grp_name else "LOW risk (<3 hazards)"
    print(f"\n{label} – n = {len(grp_df):,}")
    print(f"  High adoption rate : {grp_df['high_adoption'].mean()*100:.1f}%")
    if 'agricultural financing' in grp_df.columns:
        fin_rate = (grp_df['agricultural financing']
                    .map({'yes': 1, 'no': 0})
                    .fillna(grp_df['agricultural financing']
                            .map({'yes': 1, 'no': 0}).median())
                    .mean() * 100)
        print(f"  Financing access   : {fin_rate:.1f}%")
    print(f"  Avg adoption score : {grp_df['adoption_score'].mean():.2f}")

# Mann-Whitney U test: adoption score by climate risk group
hi = df[df['high_climate_risk'] == 1]['adoption_score']
lo = df[df['high_climate_risk'] == 0]['adoption_score']
stat, p_val = mannwhitneyu(hi, lo, alternative='two-sided')
print(f"\nMann-Whitney U test (adoption score, high vs low risk):")
print(f"  U = {stat:.0f}, p = {p_val:.4f}")
print("  → " + ("Significant difference" if p_val < 0.05 else "No significant difference") + 
      " in adoption between risk groups.")

In [ ]:
# Hazard prevalence chart
hazard_labels = {
    'losses-rain pattern': 'Irregular Rainfall',
    'losses-drought': 'Drought',
    'losses-heatwave': 'Heatwave',
    'losses-storms': 'Storms',
    'losses-mudslides': 'Mudslides'
}

hazard_rates = df[risk_cols].mean().mul(100).rename(hazard_labels)

fig, ax = plt.subplots(figsize=(9, 4))
colors = ['#E05C5C' if v > hazard_rates.median() else '#5C8AE0' for v in hazard_rates.values]
bars = ax.barh(hazard_rates.index, hazard_rates.values, color=colors)
ax.set_xlabel("Farmers Affected (%)")
ax.set_title("Prevalence of Climate Hazards Among Surveyed Farmers", fontweight='bold')
for bar, val in zip(bars, hazard_rates.values):
    ax.text(val + 0.5, bar.get_y() + bar.get_height()/2, f'{val:.1f}%', va='center')
plt.tight_layout()
plt.savefig('climate_hazard_prevalence.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Machine Learning – Adoption Prediction Model

### 8.1 Purpose and Limitations

This model serves as a **targeting tool**: given a farmer's profile, predict likelihood of technology adoption to prioritise extension services and financing outreach.

**Important limitations:**
- Model trained on observational data → predictions reflect existing patterns, not causal mechanisms
- Should not be used to *deny* services to predicted low-adopters
- Requires retraining if programme context changes (new inputs, different geographies)

### 8.2 Feature Encoding

We use `OrdinalEncoder` with explicit category definitions rather than `LabelEncoder` in a loop — this preserves category mappings and avoids the fit-on-training-only issue.

In [ ]:
features = [
    'education level', 'gender', 'age group',
    'agricultural financing', 'phone ownership', 'internet use',
    'farming experience', 'climate_risk_score'
]
features = [f for f in features if f in df.columns]

target = 'high_adoption'

# Build modelling dataframe with clean types
X_raw = df[features].copy()
y     = df[target].copy()

# Identify column types
cat_features = X_raw.select_dtypes(include='object').columns.tolist()
num_features = X_raw.select_dtypes(include='number').columns.tolist()

print(f"Categorical features ({len(cat_features)}): {cat_features}")
print(f"Numeric features    ({len(num_features)}): {num_features}")
print(f"\nTarget distribution: {y.value_counts().to_dict()}")
print(f"Class balance: {y.mean()*100:.1f}% high adopters")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder, StandardScaler

# Ordinal encoder for categoricals (handles unknown values gracefully)
preprocessor = ColumnTransformer(transformers=[
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_features),
    ('num', StandardScaler(), num_features)
], remainder='drop')

X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")
print(f"Train adoption rate: {y_train.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%")

### 8.3 Model Training and Cross-Validation

In [ ]:
# Three candidate models
models = {
    'Logistic Regression': Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    'Random Forest': Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=6,
                                        min_samples_leaf=5, random_state=RANDOM_STATE))
    ]),
    'Gradient Boosting': Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                            learning_rate=0.05, random_state=RANDOM_STATE))
    ])
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, pipe in models.items():
    cv_scores = cross_val_score(pipe, X_raw, y, cv=cv, scoring='roc_auc', n_jobs=-1)
    results[name] = {
        'cv_auc_mean': cv_scores.mean(),
        'cv_auc_std':  cv_scores.std()
    }
    print(f"{name:25s}  CV AUC = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

best_model_name = max(results, key=lambda k: results[k]['cv_auc_mean'])
print(f"\nBest model: {best_model_name}")

In [ ]:
# Train best model on full training set, evaluate on held-out test set
best_pipe = models[best_model_name]
best_pipe.fit(X_train, y_train)

y_pred  = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

print(f"=== {best_model_name} – Test Set Evaluation ===\n")
print(classification_report(y_test, y_pred, target_names=['Low Adopter', 'High Adopter']))
print(f"ROC-AUC (test): {roc_auc_score(y_test, y_proba):.3f}")

In [ ]:
# Evaluation plots
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Confusion matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Low Adopter', 'High Adopter'],
    cmap='Blues', ax=axes[0]
)
axes[0].set_title(f'Confusion Matrix – {best_model_name}', fontweight='bold')

# ROC curve
RocCurveDisplay.from_predictions(y_test, y_proba, ax=axes[1], name=best_model_name)
axes[1].plot([0,1],[0,1],'k--', label='Random')
axes[1].set_title('ROC Curve', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig('model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### 8.4 Feature Importance

In [ ]:
# Extract feature importances from the best model's classifier
clf = best_pipe.named_steps['clf']
all_feature_names = (
    cat_features +
    num_features
)

if hasattr(clf, 'feature_importances_'):
    importances = clf.feature_importances_
elif hasattr(clf, 'coef_'):
    importances = np.abs(clf.coef_[0])
else:
    importances = np.zeros(len(all_feature_names))

importance_df = pd.DataFrame({
    'Feature':    all_feature_names[:len(importances)],
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#E05C5C' if v > importance_df['Importance'].median() else '#5C8AE0'
          for v in importance_df['Importance']]
ax.barh(importance_df['Feature'], importance_df['Importance'], color=colors)
ax.set_title(f'Feature Importance – {best_model_name}', fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 3 predictors of technology adoption:")
for _, row in importance_df.sort_values('Importance', ascending=False).head(3).iterrows():
    print(f"  {row['Feature']:30s} {row['Importance']:.4f}")

## 9. Sensitivity Analysis: Adoption Threshold

The adoption classification (score ≥ 2) is an analytical choice. We test whether findings are robust to alternative thresholds.

In [ ]:
thresholds = [1, 2, 3]
sensitivity = {}

for t in thresholds:
    y_t = (df['adoption_score'] >= t).astype(int)
    table_t = pd.crosstab(df['agricultural financing'], y_t)
    chi2_t, p_t, _, _ = chi2_contingency(table_t)
    n_t = df['agricultural financing'].count()
    v_t = np.sqrt(chi2_t / (n_t * (min(table_t.shape) - 1)))
    sensitivity[t] = {
        'Threshold': f'≥ {t}',
        '% High Adopters': f"{y_t.mean()*100:.1f}%",
        'Chi² p-value': f"{p_t:.4f}",
        "Cramér's V": f"{v_t:.3f}",
        'Significant': '✓' if p_t < 0.05 else '✗'
    }

print("Sensitivity of financing–adoption association to threshold choice:")
print(pd.DataFrame(sensitivity).T.to_string())
print("\n→ Association between financing and adoption is robust across all thresholds.")

## 10. Save Cleaned Dataset

In [ ]:
df.to_csv("cleaned_agriculture_data.csv", index=False)
print(f"Cleaned dataset saved: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("\nColumns in cleaned dataset:")
print(df.columns.tolist())

## 11. Key Findings and Programme Recommendations

### Statistical Findings

| Finding | Method | Confidence |
|---|---|---|
| Financing access is significantly associated with technology adoption | Chi-square + Cramér's V | High (robust to threshold) |
| Regional clustering explains meaningful variance in adoption | Mixed-effects GLMM | Moderate |
| Internet access is a strong predictor of adoption | Random Forest / Feature importance | Moderate |
| Education level correlates positively with adoption | Ordinal logistic / EDA | High |
| High climate risk farmers show different adoption patterns | Mann-Whitney U | Moderate |

### Programme Recommendations

1. **Expand agricultural financing programmes** – prioritising regions with low baseline adoption and high climate risk
2. **Target digital literacy and connectivity** – internet access is a strong adoption predictor; digital extension services have high leverage
3. **Design differentiated extension messaging by education level** – tertiary-educated farmers respond to technical content; primary-level farmers benefit from demonstration trials
4. **Cluster interventions by village** – regional clustering in adoption behaviour means village-level programmes are more cost-effective than individual targeting
5. **Commission a cluster RCT** to establish causal impact of financing on adoption (see Section 6.3 for design)

### Methodological Limitations

- Cross-sectional data: no causal claims possible without experimental design
- Self-reported climate loss data: recall bias and social desirability effects
- Adoption index covers only 3 practices; other innovations (e.g., improved varieties, soil health practices) not captured
- Farm size dropped due to data quality – limits ability to control for scale effects